# Document Question Answering System (RAG)
### Retrieval-Augmented Generation over custom documents (PDFs / text files)

This notebook implements an end-to-end **Retrieval-Augmented Generation (RAG)** pipeline that answers
questions grounded in your own documents (notes, resumes, research papers, books, etc.) instead of
relying purely on a language model's parametric knowledge.

**Pipeline stages**

| Stage | Component |
|---|---|
| 1. Document Ingestion | PDF / TXT loader |
| 2. Text Chunking | Recursive, sentence-aware splitter with overlap |
| 3. Embedding Creation | Sentence-Transformers (with TF-IDF fallback) |
| 4. Vector Database | FAISS (with NumPy cosine-similarity fallback) + optional BM25 hybrid search |
| 5. Re-ranking (optional) | Cross-encoder re-ranker |
| 6. Query Processing | Query embedding + similarity search |
| 7. Answer Generation | Claude API (Anthropic) with local HF model fallback |

The system is designed to **degrade gracefully**: if a package or an API key is unavailable, it falls
back to a lighter-weight alternative rather than crashing, so the notebook stays runnable in restricted
environments (e.g. no internet, no API key).

> Reference implementation style inspired by: https://github.com/VivekChauhan05/RAG_Document_Question_Answering


## 0. Setup

Install the dependencies below. Everything after this cell is organized as small, testable, reusable
classes rather than a linear script - mirroring how a production RAG service would be structured
(`loader -> chunker -> embedder -> vector_store -> retriever -> generator -> pipeline`).


In [ ]:
# Core dependencies. Heavy/optional packages (torch, sentence-transformers, faiss, transformers)
# are wrapped in try/except throughout the notebook so the pipeline still runs with lighter fallbacks
# (scikit-learn TF-IDF + NumPy cosine similarity) if they are not installed or cannot be downloaded.

%pip install -q pypdf rank-bm25 scikit-learn numpy
%pip install -q sentence-transformers faiss-cpu transformers torch --quiet || echo "Optional heavy deps skipped - fallbacks will be used."
%pip install -q anthropic --quiet || echo "anthropic SDK skipped."


In [ ]:
import os
import re
import glob
import warnings
import textwrap
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Tuple

import numpy as np

warnings.filterwarnings("ignore")

# ---- Optional heavy imports (graceful fallback) ----------------------------
try:
    from sentence_transformers import SentenceTransformer, CrossEncoder
    HAS_SBERT = True
except ImportError:
    HAS_SBERT = False

try:
    import faiss
    HAS_FAISS = True
except ImportError:
    HAS_FAISS = False

try:
    from transformers import pipeline as hf_pipeline
    HAS_TRANSFORMERS = True
except ImportError:
    HAS_TRANSFORMERS = False

try:
    import anthropic
    HAS_ANTHROPIC = True
except ImportError:
    HAS_ANTHROPIC = False

from rank_bm25 import BM25Okapi
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from pypdf import PdfReader

print(f"sentence-transformers available : {HAS_SBERT}")
print(f"faiss available                 : {HAS_FAISS}")
print(f"transformers (local LLM) avail. : {HAS_TRANSFORMERS}")
print(f"anthropic SDK available         : {HAS_ANTHROPIC}")


In [ ]:
@dataclass
class RAGConfig:
    chunk_size: int = 500          # characters per chunk
    chunk_overlap: int = 80        # overlap between consecutive chunks
    top_k: int = 4                 # number of chunks to retrieve
    use_hybrid_search: bool = True # combine BM25 (keyword) + vector (semantic) search
    use_reranker: bool = True      # cross-encoder re-ranking of retrieved candidates
    embedding_model_name: str = "sentence-transformers/all-MiniLM-L6-v2"
    reranker_model_name: str = "cross-encoder/ms-marco-MiniLM-L-6-v2"
    local_llm_name: str = "google/flan-t5-base"
    anthropic_model: str = "claude-sonnet-4-6"
    anthropic_api_key_env: str = "ANTHROPIC_API_KEY"

CONFIG = RAGConfig()
CONFIG


## 1. Document Ingestion

Loads raw text out of PDFs or `.txt` files. A single file, or an entire directory of mixed
PDF/TXT files, can be ingested in one call. Each loaded document keeps its source filename as
metadata so later on every answer can be traced back to *which file* it came from.


In [ ]:
class DocumentLoader:
    """Loads PDF / TXT documents into plain text, tracking per-page and per-file provenance."""

    @staticmethod
    def load_pdf(path: str) -> List[Dict]:
        reader = PdfReader(path)
        pages = []
        for i, page in enumerate(reader.pages):
            text = page.extract_text() or ""
            text = text.strip()
            if text:
                pages.append({"text": text, "source": os.path.basename(path), "page": i + 1})
        return pages

    @staticmethod
    def load_txt(path: str) -> List[Dict]:
        with open(path, "r", encoding="utf-8", errors="ignore") as f:
            text = f.read().strip()
        return [{"text": text, "source": os.path.basename(path), "page": 1}] if text else []

    @classmethod
    def load(cls, path: str) -> List[Dict]:
        """Load a single file (.pdf or .txt)."""
        ext = os.path.splitext(path)[1].lower()
        if ext == ".pdf":
            return cls.load_pdf(path)
        elif ext in (".txt", ".md"):
            return cls.load_txt(path)
        else:
            raise ValueError(f"Unsupported file type: {ext}")

    @classmethod
    def load_directory(cls, directory: str) -> List[Dict]:
        """Load every supported file inside a directory."""
        docs = []
        for path in glob.glob(os.path.join(directory, "*")):
            if path.lower().endswith((".pdf", ".txt", ".md")):
                docs.extend(cls.load(path))
        return docs


## 2. Text Chunking

Splits raw page text into overlapping, sentence-aware chunks. Chunking matters a lot for retrieval
quality: chunks that are too large dilute the embedding with irrelevant content, while chunks that
are too small lose surrounding context. This splitter:

- Prefers to break on sentence boundaries (`. ! ?`) rather than mid-sentence
- Falls back to hard character splitting only when a single sentence exceeds `chunk_size`
- Adds a configurable character overlap between consecutive chunks so context isn't lost at chunk edges


In [ ]:
class TextChunker:
    def __init__(self, chunk_size: int = 500, chunk_overlap: int = 80):
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        self._sentence_re = re.compile(r"(?<=[.!?])\s+")

    def _split_sentences(self, text: str) -> List[str]:
        text = re.sub(r"\s+", " ", text).strip()
        return [s for s in self._sentence_re.split(text) if s]

    def chunk_text(self, text: str) -> List[str]:
        sentences = self._split_sentences(text)
        chunks, current = [], ""
        for sent in sentences:
            if len(sent) > self.chunk_size:
                # hard-split an overly long sentence
                for i in range(0, len(sent), self.chunk_size):
                    piece = sent[i:i + self.chunk_size]
                    if current:
                        chunks.append(current)
                        current = ""
                    chunks.append(piece)
                continue
            if len(current) + len(sent) + 1 <= self.chunk_size:
                current = f"{current} {sent}".strip()
            else:
                if current:
                    chunks.append(current)
                # start next chunk with overlap tail of the previous chunk
                overlap_tail = current[-self.chunk_overlap:] if current else ""
                current = f"{overlap_tail} {sent}".strip()
        if current:
            chunks.append(current)
        return chunks

    def chunk_documents(self, docs: List[Dict]) -> List[Dict]:
        """docs: list of {text, source, page} -> list of {text, source, page, chunk_id}"""
        all_chunks = []
        for doc in docs:
            for i, piece in enumerate(self.chunk_text(doc["text"])):
                all_chunks.append({
                    "text": piece,
                    "source": doc["source"],
                    "page": doc.get("page", 1),
                    "chunk_id": len(all_chunks),
                })
        return all_chunks


## 3. Embedding Model

Converts each text chunk into a dense vector that captures its semantic meaning.

- **Primary:** `sentence-transformers` (`all-MiniLM-L6-v2`) — a fast, high-quality bi-encoder.
- **Fallback:** scikit-learn `TfidfVectorizer` — a classical sparse representation that needs no
  model download, so the notebook still runs fully offline if `sentence-transformers` / model
  weights aren't available.

Both are wrapped behind the same `.encode()` interface so the rest of the pipeline doesn't need to
know which backend is active.


In [ ]:
class EmbeddingModel:
    def __init__(self, model_name: str = CONFIG.embedding_model_name):
        self.backend = None
        if HAS_SBERT:
            try:
                self.model = SentenceTransformer(model_name)
                self.backend = "sbert"
            except Exception as e:
                print(f"Could not load SentenceTransformer ({e}); falling back to TF-IDF.")
        if self.backend is None:
            self.vectorizer = TfidfVectorizer(stop_words="english")
            self.backend = "tfidf"
        print(f"[EmbeddingModel] backend = {self.backend}")

    def fit(self, corpus: List[str]):
        """Only required for the TF-IDF backend; a no-op for sentence-transformers."""
        if self.backend == "tfidf":
            self.vectorizer.fit(corpus)
        return self

    def encode(self, texts: List[str]) -> np.ndarray:
        if self.backend == "sbert":
            return np.asarray(self.model.encode(texts, show_progress_bar=False, normalize_embeddings=True))
        else:
            vecs = self.vectorizer.transform(texts).toarray()
            norms = np.linalg.norm(vecs, axis=1, keepdims=True)
            norms[norms == 0] = 1.0
            return vecs / norms


## 4. Vector Store & Hybrid Retrieval

Stores chunk embeddings for fast similarity search, and optionally combines it with **BM25**
(classic keyword search) for **hybrid search** — semantic search alone can miss exact terms
(names, numbers, acronyms) that keyword search catches, and vice versa.

- **Primary similarity search:** FAISS (`IndexFlatIP` on normalized vectors = cosine similarity)
- **Fallback similarity search:** NumPy cosine similarity (works anywhere, no compiled deps)
- **Keyword search:** BM25 (`rank_bm25`), always available
- **Fusion:** simple weighted-score fusion between the normalized BM25 and vector similarity scores


In [ ]:
class VectorStore:
    def __init__(self, embedder: EmbeddingModel, use_hybrid: bool = True):
        self.embedder = embedder
        self.use_hybrid = use_hybrid
        self.chunks: List[Dict] = []
        self.embeddings: Optional[np.ndarray] = None
        self.index = None
        self.bm25 = None

    def build(self, chunks: List[Dict]):
        self.chunks = chunks
        texts = [c["text"] for c in chunks]

        self.embedder.fit(texts)
        self.embeddings = self.embedder.encode(texts).astype("float32")

        if HAS_FAISS:
            dim = self.embeddings.shape[1]
            self.index = faiss.IndexFlatIP(dim)
            self.index.add(self.embeddings)
        # (no else branch needed — .search() falls back to NumPy if self.index is None)

        if self.use_hybrid:
            tokenized = [t.lower().split() for t in texts]
            self.bm25 = BM25Okapi(tokenized)

        return self

    def _vector_search(self, query_vec: np.ndarray, k: int) -> List[Tuple[int, float]]:
        if HAS_FAISS and self.index is not None:
            scores, idxs = self.index.search(query_vec.reshape(1, -1), k)
            return list(zip(idxs[0].tolist(), scores[0].tolist()))
        else:
            sims = cosine_similarity(query_vec.reshape(1, -1), self.embeddings)[0]
            top = np.argsort(-sims)[:k]
            return [(int(i), float(sims[i])) for i in top]

    @staticmethod
    def _normalize(scores: Dict[int, float]) -> Dict[int, float]:
        if not scores:
            return scores
        vals = np.array(list(scores.values()))
        lo, hi = vals.min(), vals.max()
        if hi - lo < 1e-9:
            return {k: 0.0 for k in scores}
        return {k: (v - lo) / (hi - lo) for k, v in scores.items()}

    def search(self, query: str, k: int = 4, alpha: float = 0.5) -> List[Dict]:
        """alpha weights vector-similarity vs BM25 when hybrid search is enabled (1.0 = vector only)."""
        query_vec = self.embedder.encode([query]).astype("float32")[0]
        vec_hits = dict(self._vector_search(query_vec, k=max(k * 3, k)))
        vec_hits_n = self._normalize(vec_hits)

        if self.use_hybrid and self.bm25 is not None:
            bm25_scores = self.bm25.get_scores(query.lower().split())
            bm25_hits = {i: bm25_scores[i] for i in vec_hits.keys()}
            bm25_hits_n = self._normalize(bm25_hits)
            fused = {i: alpha * vec_hits_n.get(i, 0.0) + (1 - alpha) * bm25_hits_n.get(i, 0.0)
                     for i in vec_hits.keys()}
        else:
            fused = vec_hits_n

        ranked = sorted(fused.items(), key=lambda x: -x[1])[:k]
        results = []
        for idx, score in ranked:
            item = dict(self.chunks[idx])
            item["score"] = float(score)
            results.append(item)
        return results


## 5. Re-ranking (optional)

Vector/BM25 retrieval is fast but approximate. A **cross-encoder** re-ranker looks at the
(query, chunk) pair *jointly* (rather than as two independently-embedded vectors) and produces a
much more accurate relevance score for the top candidates. It's slower, so it's only applied to the
small shortlist returned by the vector store, not the whole corpus. Skipped automatically if
`sentence-transformers` isn't available.


In [ ]:
class Reranker:
    def __init__(self, model_name: str = CONFIG.reranker_model_name, enabled: bool = True):
        self.enabled = enabled and HAS_SBERT
        self.model = None
        if self.enabled:
            try:
                self.model = CrossEncoder(model_name)
            except Exception as e:
                print(f"Reranker unavailable ({e}); skipping re-ranking.")
                self.enabled = False

    def rerank(self, query: str, candidates: List[Dict], top_k: int) -> List[Dict]:
        if not self.enabled or not candidates:
            return candidates[:top_k]
        pairs = [[query, c["text"]] for c in candidates]
        scores = self.model.predict(pairs)
        for c, s in zip(candidates, scores):
            c["rerank_score"] = float(s)
        return sorted(candidates, key=lambda c: -c["rerank_score"])[:top_k]


## 6. Answer Generation

Generates the final answer from the retrieved context. Three tiers, tried in order:

1. **Claude API** (`anthropic` SDK) — used if `ANTHROPIC_API_KEY` is set in the environment. Best
   quality; this is the recommended path for real use.
2. **Local Hugging Face model** (`google/flan-t5-base`) — a small instruction-tuned model that runs
   fully offline once downloaded, used if no API key is present.
3. **Extractive fallback** — if neither an API key nor a local model is available, the system simply
   returns the highest-scoring retrieved passage(s) verbatim. This guarantees the pipeline *always*
   produces a grounded answer, even with zero external dependencies.

The prompt explicitly instructs the model to answer **only** from the provided context and to say
so when the answer isn't contained in the documents — this is what keeps the system's answers
grounded and reduces hallucination.


In [ ]:
PROMPT_TEMPLATE = """You are a helpful assistant answering questions using ONLY the context below,
which was retrieved from the user's own documents. If the answer is not contained in the context,
say you don't have enough information in the provided documents - do not make anything up.

Context:
{context}

Question: {question}

Answer (concise, and cite the source file/page in brackets where relevant):"""


class AnswerGenerator:
    def __init__(self, config: RAGConfig = CONFIG):
        self.config = config
        self.mode = None
        self.client = None
        self.local_llm = None

        api_key = os.environ.get(config.anthropic_api_key_env)
        if HAS_ANTHROPIC and api_key:
            self.client = anthropic.Anthropic(api_key=api_key)
            self.mode = "anthropic"
        elif HAS_TRANSFORMERS:
            try:
                self.local_llm = hf_pipeline("text2text-generation", model=config.local_llm_name)
                self.mode = "local_llm"
            except Exception as e:
                print(f"Local LLM unavailable ({e}); using extractive fallback.")
                self.mode = "extractive"
        else:
            self.mode = "extractive"
        print(f"[AnswerGenerator] mode = {self.mode}")

    @staticmethod
    def _format_context(chunks: List[Dict]) -> str:
        parts = []
        for c in chunks:
            tag = f"[{c['source']} p.{c.get('page', 1)}]"
            parts.append(f"{tag} {c['text']}")
        return "\n\n".join(parts)

    def generate(self, question: str, chunks: List[Dict]) -> str:
        context = self._format_context(chunks)
        prompt = PROMPT_TEMPLATE.format(context=context, question=question)

        if self.mode == "anthropic":
            resp = self.client.messages.create(
                model=self.config.anthropic_model,
                max_tokens=512,
                messages=[{"role": "user", "content": prompt}],
            )
            return "".join(b.text for b in resp.content if b.type == "text").strip()

        elif self.mode == "local_llm":
            out = self.local_llm(prompt, max_new_tokens=200, do_sample=False)
            return out[0]["generated_text"].strip()

        else:  # extractive fallback
            best = chunks[0]
            return (f"(Extractive fallback - no LLM available) Most relevant passage "
                    f"[{best['source']} p.{best.get('page', 1)}]:\n{best['text']}")


## 7. The Full Pipeline

`RAGPipeline` wires every stage above into two public methods:

- `ingest(paths)` — load -> chunk -> embed -> index
- `query(question)` — embed query -> hybrid retrieve -> re-rank -> generate answer

This is the class an application (a CLI, a Streamlit app, an API endpoint) would actually import
and call.


In [ ]:
class RAGPipeline:
    def __init__(self, config: RAGConfig = CONFIG):
        self.config = config
        self.chunker = TextChunker(config.chunk_size, config.chunk_overlap)
        self.embedder = EmbeddingModel(config.embedding_model_name)
        self.store = VectorStore(self.embedder, use_hybrid=config.use_hybrid_search)
        self.reranker = Reranker(config.reranker_model_name, enabled=config.use_reranker)
        self.generator = AnswerGenerator(config)
        self.chunks: List[Dict] = []

    def ingest(self, paths: List[str]):
        """paths: list of file paths and/or directory paths."""
        docs = []
        for p in paths:
            if os.path.isdir(p):
                docs.extend(DocumentLoader.load_directory(p))
            else:
                docs.extend(DocumentLoader.load(p))
        if not docs:
            raise ValueError("No documents were loaded - check the given paths.")

        self.chunks = self.chunker.chunk_documents(docs)
        self.store.build(self.chunks)
        print(f"Ingested {len(docs)} page(s) from {len(paths)} source(s) -> {len(self.chunks)} chunks indexed.")
        return self

    def retrieve(self, question: str) -> List[Dict]:
        candidates = self.store.search(question, k=max(self.config.top_k * 3, self.config.top_k))
        return self.reranker.rerank(question, candidates, top_k=self.config.top_k)

    def query(self, question: str, verbose: bool = True) -> Dict:
        top_chunks = self.retrieve(question)
        answer = self.generator.generate(question, top_chunks)

        if verbose:
            print(f"Q: {question}\n")
            print(f"A: {answer}\n")
            print("Sources used:")
            for c in top_chunks:
                snippet = textwrap.shorten(c["text"], width=100)
                score_key = "rerank_score" if "rerank_score" in c else "score"
                print(f"  - [{c['source']} p.{c.get('page', 1)}] (score={c.get(score_key, 0):.3f}) {snippet}")

        return {"question": question, "answer": answer, "sources": top_chunks}


## 8. Demo Run

A small self-contained sample document is generated below so the notebook produces a working demo
out of the box. **To use your own data**, just point `SOURCE_PATHS` at your PDF/TXT file(s) or a
folder containing them, e.g.:

```python
SOURCE_PATHS = ["/mnt/user-data/uploads/my_resume.pdf"]
# or
SOURCE_PATHS = ["/mnt/user-data/uploads/"]   # ingest every PDF/TXT in the folder
```


In [ ]:
SAMPLE_DOC = """
LegalArena: A Multi-Agent Adversarial Legal Reasoning System

LegalArena is a final year project in the Agentic AI domain. The system simulates a courtroom
debate using five specialized agents: a Prosecutor agent, a Defense agent, a Judge agent, a
Witness agent, and a Clerk agent that manages case records. Each agent is powered by a large
language model and communicates through a structured debate loop, where arguments are exchanged
over multiple rounds before the Judge agent issues a ruling.

The frontend is built with Next.js and Tailwind CSS, using the Vercel AI SDK to stream agent
responses in real time to the user. The backend is implemented in Python using FastAPI, which
orchestrates the multi-agent debate loop and exposes REST and WebSocket endpoints consumed by the
frontend.

The project is structured across two semesters. In the first semester, the focus is on building
the core agent architecture, the debate loop, and a baseline single-case demo. In the second
semester, the focus shifts to evaluation, handling more complex multi-issue cases, adding
retrieval-augmented grounding in real case law, and polishing the user interface.

Retrieval-Augmented Generation (RAG) is used within LegalArena to ground each agent's arguments in
retrieved statutes and precedent, rather than allowing the agents to rely purely on the language
model's internal knowledge. This reduces hallucinated case citations and improves the factual
grounding of the courtroom debate.
"""

os.makedirs("sample_data", exist_ok=True)
with open("sample_data/legalarena_notes.txt", "w") as f:
    f.write(SAMPLE_DOC)

SOURCE_PATHS = ["sample_data/legalarena_notes.txt"]   # <-- point this at your own PDFs/TXT files

pipeline = RAGPipeline(CONFIG)
pipeline.ingest(SOURCE_PATHS)


In [ ]:
questions = [
    "What agents make up the LegalArena system?",
    "What frontend and backend technologies are used?",
    "Why is RAG used inside LegalArena?",
    "What happens in the second semester of the project?",
]

results = []
for q in questions:
    res = pipeline.query(q)
    results.append(res)
    print("=" * 90)


## 9. Evaluation

A minimal retrieval evaluation harness: given a set of `(question, expected_keyword)` pairs, it
checks whether at least one retrieved chunk contains the expected keyword — a quick proxy for
**retrieval accuracy** (a full evaluation would use labeled relevant-chunk IDs and compute metrics
like Precision@k / Recall@k / MRR, or use an LLM-as-judge for answer correctness).


In [ ]:
def evaluate_retrieval(pipeline: RAGPipeline, cases: List[Tuple[str, str]]) -> float:
    """cases: list of (question, keyword_expected_in_a_retrieved_chunk)"""
    hits = 0
    for question, keyword in cases:
        retrieved = pipeline.retrieve(question)
        found = any(keyword.lower() in c["text"].lower() for c in retrieved)
        hits += int(found)
        print(f"[{'HIT ' if found else 'MISS'}] '{question}' -> expected keyword '{keyword}'")
    accuracy = hits / len(cases)
    print(f"\nRetrieval hit-rate: {accuracy:.0%} ({hits}/{len(cases)})")
    return accuracy

eval_cases = [
    ("Who manages case records?", "Clerk"),
    ("What UI framework is used?", "Tailwind"),
    ("How many semesters does the project span?", "two semesters"),
]
_ = evaluate_retrieval(pipeline, eval_cases)


## 10. Improvements & Experiments

- **Chunking:** try semantic/structure-aware chunking (split on headings, tables) instead of fixed character windows.
- **Embeddings:** compare `all-MiniLM-L6-v2` against larger models (`bge-large`, `e5-large`) for domain-specific accuracy vs. latency trade-offs.
- **Retrieval:** tune the hybrid-search `alpha` weight; add query expansion / HyDE (hypothetical document embeddings).
- **Re-ranking:** benchmark cross-encoder re-ranking vs. no re-ranking on the evaluation harness above.
- **Generation:** try different LLMs (Claude Sonnet vs. Haiku vs. local models) and compare answer groundedness.
- **Evaluation:** move from keyword-hit checks to a proper labeled eval set with Precision@k/Recall@k/MRR, plus LLM-as-judge scoring for answer faithfulness.

## 11. Conclusion

This notebook implements a complete, modular RAG pipeline — ingestion, chunking, embedding,
hybrid vector+keyword retrieval, optional re-ranking, and grounded answer generation with
multi-tier fallbacks — that answers questions over custom, private documents rather than relying on
a model's parametric knowledge. The same architecture generalizes to chatbots, knowledge
assistants, enterprise search, and documentation Q&A tools; swapping in a production vector database
(e.g. Pinecone, Weaviate, Chroma) and a larger document set is a drop-in extension of the
`VectorStore` and `DocumentLoader` classes defined here.
